# 03 — Queries de verificación

Este notebook contiene consultas analíticas sobre el modelo de datos de **SkeletIA** almacenado en Google BigQuery.

El objetivo es comprobar que las relaciones entre las distintas tablas funcionan correctamente y demostrar que el modelo permite responder preguntas relevantes para el negocio.

Las consultas cubren distintas áreas:

- ventas e ingresos
- productos y categorías
- clientes y geografía
- adquisición de clientes
- logística
- pagos
- rentabilidad
- comportamiento de compra
- valoraciones
- stock

Para los análisis de ventas e ingresos se consideran, salvo que se indique lo contrario, únicamente los pedidos con estado `delivered`.

## Conexión con BigQuery

In [1]:
from pathlib import Path
from dotenv import load_dotenv
from google.cloud import bigquery
import os
import pandas as pd


# Localizamos la raíz del proyecto.
ROOT_DIR = Path.cwd()

if not (ROOT_DIR / ".env").exists():
    ROOT_DIR = ROOT_DIR.parent.parent


# Cargamos las variables de entorno.
ENV_PATH = ROOT_DIR / ".env"

load_dotenv(ENV_PATH)


# Recuperamos la configuración.
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

credentials_path = Path(os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))


# Si la ruta de credenciales es relativa,
# la hacemos relativa a la raíz del proyecto.
if not credentials_path.is_absolute():
    credentials_path = ROOT_DIR / credentials_path


credentials_path = credentials_path.resolve()

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path)


# Creamos el cliente de BigQuery.
client = bigquery.Client(project=PROJECT_ID)


print(f"Proyecto: {PROJECT_ID}")
print(f"Dataset: {DATASET_ID}")

Proyecto: tc-sql-bometon
Dataset: skeletia


## Query 1 — Ingresos mensuales

### Pregunta de negocio

¿Cómo ha evolucionado la facturación de SkeletIA mes a mes?

In [2]:
QUERY_01 = f"""
WITH order_totals AS (

    SELECT
        o.order_id,
        o.order_date,
        o.shipping_cost,
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS products_revenue

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    WHERE
        o.status = 'delivered'

    GROUP BY
        o.order_id,
        o.order_date,
        o.shipping_cost
)


SELECT
    DATE_TRUNC(DATE(order_date), MONTH) AS month,
    COUNT(*) AS delivered_orders,
    ROUND(SUM(products_revenue), 2) AS products_revenue,
    ROUND(SUM(shipping_cost), 2) AS shipping_revenue,
    ROUND(
        SUM(products_revenue + shipping_cost), 2
    ) AS total_revenue,
    ROUND(
        AVG(products_revenue + shipping_cost), 2
    ) AS average_order_value

FROM
    order_totals

GROUP BY
    month

ORDER BY
    month
"""

In [3]:
df_query_01 = client.query(QUERY_01).to_dataframe()
df_query_01

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,month,delivered_orders,products_revenue,shipping_revenue,total_revenue,average_order_value
0,2025-03-01,27,51904.980000000,90.850000000,51995.830000000,1925.770000000
1,2025-04-01,21,37527.040000000,94.870000000,37621.910000000,1791.520000000
2,2025-05-01,28,54943.180000000,161.770000000,55104.950000000,1968.030000000
3,2025-06-01,38,71243.560000000,165.760000000,71409.320000000,1879.190000000
4,2025-07-01,50,83993.890000000,207.700000000,84201.590000000,1684.030000000
5,2025-08-01,50,88617.370000000,240.660000000,88858.030000000,1777.160000000
6,2025-09-01,34,50969.040000000,182.730000000,51151.770000000,1504.460000000
7,2025-10-01,62,107265.400000000,266.610000000,107532.010000000,1734.390000000
8,2025-11-01,58,96402.750000000,219.680000000,96622.430000000,1665.900000000
9,2025-12-01,75,146989.990000000,299.570000000,147289.560000000,1963.860000000


## Query 2 — Productos más vendidos

### Pregunta de negocio

¿Cuáles son los productos con mayor volumen de ventas de SkeletIA?

In [4]:
QUERY_02 = f"""
SELECT
    p.product_id,
    p.sku,
    p.product_name,
    b.brand_name,
    c.category_name,
    COUNT(DISTINCT o.order_id) AS delivered_orders,
    SUM(oi.quantity) AS units_sold,
    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ), 2
    ) AS revenue,
    ROUND(
        SAFE_DIVIDE(
            SUM(
                oi.quantity
                * oi.unit_price
                * (1 - oi.discount_percent / 100)
            ),
            SUM(oi.quantity)
        ),
        2
    ) AS average_effective_unit_price

FROM
    `{PROJECT_ID}.{DATASET_ID}.orders` AS o

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
    ON o.order_id = oi.order_id

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.products` AS p
    ON oi.product_id = p.product_id

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.brands` AS b
    ON p.brand_id = b.brand_id

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.categories` AS c
    ON p.category_id = c.category_id

WHERE
    o.status = 'delivered'

GROUP BY
    p.product_id,
    p.sku,
    p.product_name,
    b.brand_name,
    c.category_name

ORDER BY
    units_sold DESC,
    revenue DESC

LIMIT 10
"""

In [5]:
df_query_02 = client.query(QUERY_02).to_dataframe()
df_query_02

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_id,sku,product_name,brand_name,category_name,delivered_orders,units_sold,revenue,average_effective_unit_price
0,14,SKL-07-0014,Pope Group A774,Pope Group,Smart Home,53,70,5571.980000000,79.600000000
1,24,SKL-07-0024,Alvarez Inc M129,Alvarez Inc,Smart Home,54,69,6798.690000000,98.530000000
2,70,SKL-02-0070,Reed-Bass S923,Reed-Bass,Laptops,51,68,116916.450000000,1719.360000000
3,21,SKL-06-0021,"Edwards, Murphy and Escobar V594","Edwards, Murphy and Escobar",Wearables,51,68,17335.700000000,254.940000000
4,58,SKL-07-0058,"Grant, Oneill and Medina M888","Grant, Oneill and Medina",Smart Home,50,66,17245.080000000,261.290000000
5,30,SKL-03-0030,Pope Group S668,Pope Group,Tablets,50,65,53432.550000000,822.040000000
6,12,SKL-04-0012,"Grant, Oneill and Medina M473","Grant, Oneill and Medina",Audio,53,64,14280.930000000,223.140000000
7,49,SKL-08-0049,SkeletIA X980,SkeletIA,Storage & Components,52,64,2790.460000000,43.600000000
8,60,SKL-07-0060,Johnson-Carter V650,Johnson-Carter,Smart Home,42,63,20696.770000000,328.520000000
9,64,SKL-03-0064,Pope Group A270,Pope Group,Tablets,49,62,72611.160000000,1171.150000000


## Query 3 — Clientes por país

### Pregunta de negocio

¿Cómo se distribuyen geográficamente los clientes de SkeletIA?

In [6]:
QUERY_03 = f"""
SELECT
    co.country_id,
    co.country_code,
    co.country_name,
    COUNT(cu.customer_id) AS total_customers,
    COUNTIF(cu.is_active) AS active_customers,
    ROUND(
        SAFE_DIVIDE(COUNTIF(cu.is_active), COUNT(cu.customer_id)) * 100,
        2
    ) AS active_customers_pct,
    COUNT(DISTINCT ci.city_id) AS cities_with_customers,
    DATE_FROM_UNIX_DATE(CAST(AVG(UNIX_DATE(DATE(cu.registered_at))) AS INT64)) AS average_registration_date

FROM
    `{PROJECT_ID}.{DATASET_ID}.customers` AS cu

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.cities` AS ci
    ON cu.city_id = ci.city_id

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.countries` AS co
    ON ci.country_id = co.country_id

GROUP BY
    co.country_id,
    co.country_code,
    co.country_name

ORDER BY
    total_customers DESC
"""

In [7]:
df_query_03 = client.query(QUERY_03).to_dataframe()
df_query_03

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,country_id,country_code,country_name,total_customers,active_customers,active_customers_pct,cities_with_customers,average_registration_date
0,5,PT,Portugal,76,72,94.74,3,2025-07-19
1,6,NL,Netherlands,67,66,98.51,3,2025-06-05
2,7,BE,Belgium,66,66,100.00,3,2025-06-13
3,3,DE,Germany,65,64,98.46,3,2025-06-05
4,4,IT,Italy,61,60,98.36,3,2025-05-13
5,2,FR,France,56,54,96.43,3,2025-07-14
6,8,AT,Austria,55,54,98.18,3,2025-06-13
7,1,ES,Spain,54,52,96.30,3,2025-05-16


## Query 4 — Tiempos de preparación y entrega

### Pregunta de negocio

¿Cuánto tarda SkeletIA en procesar y entregar sus pedidos?

In [8]:
QUERY_04 = f"""
SELECT
    COUNT(*) AS delivered_orders,
    ROUND(AVG(DATE_DIFF(DATE(shipped_at), DATE(order_date), DAY)), 2) AS avg_processing_days,
    ROUND(AVG(DATE_DIFF(DATE(delivered_at), DATE(shipped_at), DAY)), 2) AS avg_shipping_days,
    ROUND(AVG(DATE_DIFF(DATE(delivered_at), DATE(order_date), DAY)), 2) AS avg_total_delivery_days,
    APPROX_QUANTILES(
        DATE_DIFF(DATE(delivered_at), DATE(order_date), DAY), 100
    )[OFFSET(50)] AS median_delivery_days,
    APPROX_QUANTILES(
        DATE_DIFF(DATE(delivered_at), DATE(order_date), DAY), 100
    )[OFFSET(90)] AS p90_delivery_days,
    MIN(DATE_DIFF(DATE(delivered_at), DATE(order_date), DAY)) AS min_delivery_days,
    MAX(DATE_DIFF(DATE(delivered_at), DATE(order_date), DAY)) AS max_delivery_days

FROM
    `{PROJECT_ID}.{DATASET_ID}.orders`

WHERE
    status = 'delivered'
    AND shipped_at IS NOT NULL
    AND delivered_at IS NOT NULL
"""

In [9]:
df_query_04 = client.query(QUERY_04).to_dataframe()
df_query_04

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,delivered_orders,avg_processing_days,avg_shipping_days,avg_total_delivery_days,median_delivery_days,p90_delivery_days,min_delivery_days,max_delivery_days
0,1311,2.02,3.01,5.04,5,7,2,8


## Query 5 — Ingresos, costes y margen bruto por categoría

### Pregunta de negocio

¿Qué categorías de productos generan más ingresos y cuáles son realmente más rentables?

In [10]:
QUERY_05 = f"""
WITH category_metrics AS (

    SELECT
        c.category_id,
        c.category_name,
        COUNT(DISTINCT o.order_id) AS delivered_orders,
        SUM(oi.quantity) AS units_sold,
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue,
        SUM(oi.quantity * oi.unit_cost) AS cost

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.products` AS p
        ON oi.product_id = p.product_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.categories` AS c
        ON p.category_id = c.category_id

    WHERE
        o.status = 'delivered'

    GROUP BY
        c.category_id,
        c.category_name
)


SELECT
    category_name,
    delivered_orders,
    units_sold,
    ROUND(revenue, 2) AS revenue,
    ROUND(cost, 2) AS cost,
    ROUND(revenue - cost, 2) AS gross_margin,
    ROUND(
        SAFE_DIVIDE(revenue - cost, revenue) * 100, 2
    ) AS gross_margin_pct

FROM
    category_metrics

ORDER BY
    gross_margin DESC
"""

In [11]:
df_query_05 = client.query(QUERY_05).to_dataframe()
df_query_05

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_name,delivered_orders,units_sold,revenue,cost,gross_margin,gross_margin_pct
0,Laptops,575,855,1187950.510000000,791090.250000000,396860.260000000,33.410000000
1,Tablets,327,469,386931.220000000,270746.040000000,116185.180000000,30.030000000
2,Storage & Components,457,639,248418.590000000,164081.150000000,84337.440000000,33.950000000
3,Smart Home,410,610,125924.920000000,81703.150000000,44221.770000000,35.120000000
4,Smartphones,184,227,154876.380000000,112076.730000000,42799.650000000,27.630000000
5,Wearables,281,406,109759.240000000,72236.370000000,37522.870000000,34.190000000
6,Peripherals,198,261,58561.590000000,39069.970000000,19491.620000000,33.280000000
7,Audio,164,213,43366.370000000,32894.490000000,10471.880000000,24.150000000


## Query 6 — Top 3 productos por facturación dentro de cada categoría

### Pregunta de negocio

¿Cuáles son los productos que más facturación generan dentro de cada categoría?

In [12]:
QUERY_06 = f"""
WITH product_sales AS (
    SELECT
        c.category_id,
        c.category_name,
        p.product_id,
        p.sku,
        p.product_name,
        b.brand_name,
        COUNT(DISTINCT o.order_id) AS delivered_orders,
        SUM(oi.quantity) AS units_sold,
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.products` AS p
        ON oi.product_id = p.product_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.categories` AS c
        ON p.category_id = c.category_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.brands` AS b
        ON p.brand_id = b.brand_id

    WHERE
        o.status = 'delivered'

    GROUP BY
        c.category_id,
        c.category_name,
        p.product_id,
        p.sku,
        p.product_name,
        b.brand_name
),


ranked_products AS (
    SELECT
        *,
        DENSE_RANK() OVER (PARTITION BY category_id ORDER BY revenue DESC) AS revenue_rank

    FROM
        product_sales
)


SELECT
    category_name,
    revenue_rank,
    product_id,
    sku,
    product_name,
    brand_name,
    delivered_orders,
    units_sold,
    ROUND(revenue, 2) AS revenue

FROM
    ranked_products

WHERE
    revenue_rank <= 3

ORDER BY
    category_name,
    revenue_rank,
    revenue DESC
"""

In [13]:
df_query_06 = client.query(QUERY_06).to_dataframe()
df_query_06

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_name,revenue_rank,product_id,sku,product_name,brand_name,delivered_orders,units_sold,revenue
0,Audio,1,40,SKL-04-0040,Alvarez Inc M601,Alvarez Inc,49,60,23966.930000000
1,Audio,2,12,SKL-04-0012,"Grant, Oneill and Medina M473","Grant, Oneill and Medina",53,64,14280.930000000
2,Audio,3,35,SKL-04-0035,Johnson-Carter S404,Johnson-Carter,32,35,2734.540000000
3,Laptops,1,4,SKL-02-0004,SkeletIA A204,SkeletIA,44,55,118729.870000000
4,Laptops,2,70,SKL-02-0070,Reed-Bass S923,Reed-Bass,51,68,116916.450000000
5,Laptops,3,57,SKL-02-0057,Reed-Bass A424,Reed-Bass,47,56,116718.920000000
6,Peripherals,1,36,SKL-05-0036,"Grant, Oneill and Medina X894","Grant, Oneill and Medina",44,60,16865.180000000
7,Peripherals,2,55,SKL-05-0055,SkeletIA A310,SkeletIA,48,56,15285.830000000
8,Peripherals,3,67,SKL-05-0067,Adams Ltd X333,Adams Ltd,37,43,12413.350000000
9,Smart Home,1,60,SKL-07-0060,Johnson-Carter V650,Johnson-Carter,42,63,20696.770000000


## Query 7 — Segmentación de clientes por valor

### Pregunta de negocio

¿Qué clientes aportan más valor a SkeletIA y cómo podemos segmentarlos según su comportamiento de compra?

In [14]:
QUERY_07 = f"""
WITH order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_date,
        o.shipping_cost,
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) + o.shipping_cost AS order_total

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    WHERE
        o.status = 'delivered'

    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_date,
        o.shipping_cost
),


customer_metrics AS (
    SELECT
        cu.customer_id,
        cu.first_name,
        cu.last_name,
        cu.email,
        COUNT(ot.order_id) AS delivered_orders,
        COALESCE(SUM(ot.order_total), 0) AS total_spent,
        COALESCE(AVG(ot.order_total), 0) AS average_order_value,
        MAX(ot.order_date) AS last_order_date

    FROM
        `{PROJECT_ID}.{DATASET_ID}.customers` AS cu

    LEFT JOIN
        order_totals AS ot
        ON cu.customer_id = ot.customer_id

    GROUP BY
        cu.customer_id,
        cu.first_name,
        cu.last_name,
        cu.email
),


spending_threshold AS (
    SELECT
        APPROX_QUANTILES(total_spent, 100)[OFFSET(75)] AS p75_spending

    FROM
        customer_metrics

    WHERE
        delivered_orders > 0
)


SELECT
    cm.customer_id,
    CONCAT(cm.first_name, ' ', cm.last_name) AS customer_name,
    cm.email,
    cm.delivered_orders,
    ROUND(cm.total_spent, 2) AS total_spent,
    ROUND(cm.average_order_value, 2) AS average_order_value,
    DATE(cm.last_order_date) AS last_order_date,

    CASE
        WHEN cm.last_order_date IS NULL THEN NULL
        ELSE DATE_DIFF(CURRENT_DATE(), DATE(cm.last_order_date), DAY)
    END AS days_since_last_order,

    ROUND(st.p75_spending, 2) AS p75_spending,

    CASE
        WHEN cm.delivered_orders = 0 THEN 'No purchases'
        WHEN cm.total_spent >= st.p75_spending THEN 'High value'
        WHEN cm.delivered_orders >= 3 THEN 'Recurring'
        ELSE 'Occasional'
    END AS customer_segment

FROM
    customer_metrics AS cm

CROSS JOIN
    spending_threshold AS st

ORDER BY
    cm.total_spent DESC
"""

In [15]:
df_query_07 = client.query(QUERY_07).to_dataframe()
df_query_07

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,customer_name,email,delivered_orders,total_spent,average_order_value,last_order_date,days_since_last_order,p75_spending,customer_segment
0,395,Stephanie Hiller,vhartung.0395@mail.com,8,21537.820000000,2692.230000000,2026-05-12,116,6777.730000000,High value
1,76,David Pichon,jacqueline46.0076@icloud.com,6,19919.690000000,3319.950000000,2026-08-02,34,6777.730000000,High value
2,84,Filippa Calarco,rpizzamano.0084@icloud.com,2,19028.460000000,9514.230000000,2026-08-13,23,6777.730000000,High value
3,103,Laurens Greij,lukamahieu.0103@icloud.com,5,17760.650000000,3552.130000000,2026-07-20,47,6777.730000000,High value
4,333,Pasquale Tremonti,lara90.0333@mail.com,5,17398.490000000,3479.700000000,2026-08-03,33,6777.730000000,High value
...,...,...,...,...,...,...,...,...,...,...
495,239,Paola Jadot,bdocquier.0239@gmail.com,0,0E-9,0E-9,NaT,<NA>,6777.730000000,No purchases
496,106,Morena Basadonna,ufarinelli.0106@proton.me,0,0E-9,0E-9,NaT,<NA>,6777.730000000,No purchases
497,150,Cato Willems,evie73.0150@gmail.com,0,0E-9,0E-9,NaT,<NA>,6777.730000000,No purchases
498,32,Türkan Stadelmann,emilia51.0032@mail.com,0,0E-9,0E-9,NaT,<NA>,6777.730000000,No purchases


## Query 8 — Tasa de repetición de compra por canal de adquisición

### Pregunta de negocio

¿Qué canales de adquisición generan clientes con mayor fidelidad y repetición de compra?

In [16]:
QUERY_08 = f"""
WITH order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) + o.shipping_cost AS order_total

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    WHERE
        o.status = 'delivered'

    GROUP BY
        o.order_id,
        o.customer_id,
        o.shipping_cost
),

customer_purchase_metrics AS (
    SELECT
        cu.customer_id,
        cu.channel_id,
        COUNT(ot.order_id) AS delivered_orders,
        COALESCE(SUM(ot.order_total), 0) AS total_spent

    FROM
        `{PROJECT_ID}.{DATASET_ID}.customers` AS cu

    LEFT JOIN
        order_totals AS ot
        ON cu.customer_id = ot.customer_id

    GROUP BY
        cu.customer_id,
        cu.channel_id
)


SELECT
    ac.channel_code,
    ac.channel_name,
    COUNT(cpm.customer_id) AS total_customers,
    COUNTIF(cpm.delivered_orders >= 1) AS purchasing_customers,
    COUNTIF(cpm.delivered_orders >= 2) AS repeat_customers,
    ROUND(
        SAFE_DIVIDE(COUNTIF(cpm.delivered_orders >= 1), COUNT(cpm.customer_id)) * 100,
        2
    ) AS purchaser_rate_pct,
    ROUND(
        SAFE_DIVIDE(COUNTIF(cpm.delivered_orders >= 2), COUNTIF(cpm.delivered_orders >= 1)) * 100,
        2
    ) AS repeat_purchase_rate_pct,

    ROUND(
        AVG(
            CASE
                WHEN cpm.delivered_orders >= 1
                THEN cpm.delivered_orders
                ELSE NULL
            END
        ),
        2
    ) AS avg_orders_per_purchasing_customer,

    ROUND(
        AVG(
            CASE
                WHEN cpm.delivered_orders >= 1
                THEN cpm.total_spent
                ELSE NULL
            END
        ),
        2
    ) AS avg_spend_per_purchasing_customer

FROM
    customer_purchase_metrics AS cpm

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.acquisition_channels` AS ac
    ON cpm.channel_id = ac.channel_id

GROUP BY
    ac.channel_id,
    ac.channel_code,
    ac.channel_name

ORDER BY
    repeat_purchase_rate_pct DESC,
    avg_spend_per_purchasing_customer DESC
"""

In [17]:
df_query_08 = client.query(QUERY_08).to_dataframe()
df_query_08

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,channel_code,channel_name,total_customers,purchasing_customers,repeat_customers,purchaser_rate_pct,repeat_purchase_rate_pct,avg_orders_per_purchasing_customer,avg_spend_per_purchasing_customer
0,affiliate,Affiliate,33,31,26,93.94,83.87,2.87,5331.350000000
1,referral,Referral,61,59,49,96.72,83.05,3.20,5617.600000000
2,social_media,Social media,97,95,76,97.94,80.00,2.79,4772.430000000
3,organic,Organic search,185,170,131,91.89,77.06,2.68,4850.720000000
4,paid_ads,Paid ads,124,113,85,91.13,75.22,2.76,4840.500000000


## Query 9 — Tiempo hasta la primera compra por canal de adquisición

### Pregunta de negocio

¿Qué canales de adquisición consiguen que los clientes realicen su primera compra más rápidamente?

In [18]:
QUERY_09 = f"""
WITH customer_first_order AS (
    SELECT
        cu.customer_id,
        cu.channel_id,
        cu.registered_at,
        MIN(o.order_date) AS first_order_date

    FROM
        `{PROJECT_ID}.{DATASET_ID}.customers` AS cu

    LEFT JOIN
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o
        ON cu.customer_id = o.customer_id

    GROUP BY
        cu.customer_id,
        cu.channel_id,
        cu.registered_at
),

customer_conversion AS (
    SELECT
        customer_id,
        channel_id,
        registered_at,
        first_order_date,

        CASE
            WHEN first_order_date IS NOT NULL
            THEN DATE_DIFF(DATE(first_order_date), DATE(registered_at), DAY)
            ELSE NULL
        END AS days_to_first_order

    FROM
        customer_first_order
)


SELECT
    ac.channel_code,
    ac.channel_name,
    COUNT(cc.customer_id) AS total_customers,
    COUNTIF(cc.first_order_date IS NOT NULL) AS customers_with_order,
    ROUND(
        SAFE_DIVIDE(COUNTIF(cc.first_order_date IS NOT NULL), COUNT(cc.customer_id)) * 100, 2
    ) AS conversion_pct,
    ROUND(AVG(cc.days_to_first_order), 2) AS avg_days_to_first_order,
    APPROX_QUANTILES(cc.days_to_first_order, 100 IGNORE NULLS)[OFFSET(50)] AS median_days_to_first_order,
    MIN(cc.days_to_first_order) AS min_days_to_first_order,
    MAX(cc.days_to_first_order) AS max_days_to_first_order

FROM
    customer_conversion AS cc

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.acquisition_channels` AS ac
    ON cc.channel_id = ac.channel_id

GROUP BY
    ac.channel_id,
    ac.channel_code,
    ac.channel_name

ORDER BY
    avg_days_to_first_order ASC
"""

In [19]:
df_query_09 = client.query(QUERY_09).to_dataframe()
df_query_09

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,channel_code,channel_name,total_customers,customers_with_order,conversion_pct,avg_days_to_first_order,median_days_to_first_order,min_days_to_first_order,max_days_to_first_order
0,paid_ads,Paid ads,124,122,98.39,118.70,59,0,621
1,affiliate,Affiliate,33,33,100.00,129.15,80,1,489
2,referral,Referral,61,60,98.36,133.87,69,2,505
3,social_media,Social media,97,95,97.94,159.38,96,0,628
4,organic,Organic search,185,180,97.30,170.08,97,0,749


## Query 10 — Rendimiento de los métodos de pago

### Pregunta de negocio

¿Qué métodos de pago utiliza más la clientela de SkeletIA y qué rendimiento presenta cada uno?

In [20]:
QUERY_10 = f"""
SELECT
    payment_method,
    COUNT(*) AS total_payments,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS average_payment_amount,
    COUNTIF(status = 'completed') AS completed_payments,
    COUNTIF(status = 'pending') AS pending_payments,
    COUNTIF(status = 'failed') AS failed_payments,
    COUNTIF(status = 'refunded') AS refunded_payments,
    ROUND(SAFE_DIVIDE(COUNTIF(status = 'completed'), COUNT(*)) * 100, 2) AS completed_pct,
    ROUND(SAFE_DIVIDE(COUNTIF(status = 'failed'), COUNT(*)) * 100, 2) AS failed_pct,
    ROUND(SAFE_DIVIDE(COUNTIF(status = 'refunded'), COUNT(*)) * 100, 2) AS refunded_pct

FROM
    `{PROJECT_ID}.{DATASET_ID}.payments`

GROUP BY
    payment_method

ORDER BY
    total_payments DESC,
    total_amount DESC
"""

In [21]:
df_query_10 = client.query(QUERY_10).to_dataframe()
df_query_10

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,payment_method,total_payments,total_amount,average_payment_amount,completed_payments,pending_payments,failed_payments,refunded_payments,completed_pct,failed_pct,refunded_pct
0,samsung_pay,974,1709345.200000000,1754.970000000,798,54,21,101,81.93,2.16,10.37
1,cash_on_delivery,463,830452.040000000,1793.630000000,384,19,4,56,82.94,0.86,12.10
2,google_pay,251,427059.290000000,1701.430000000,212,9,3,27,84.46,1.20,10.76
3,card,145,251408.350000000,1733.850000000,126,8,1,10,86.90,0.69,6.90
4,bank_transfer,72,139143.530000000,1932.550000000,53,7,1,11,73.61,1.39,15.28
5,paypal,67,124896.150000000,1864.120000000,56,1,2,8,83.58,2.99,11.94
6,apple_pay,28,36750.410000000,1312.510000000,21,2,0,5,75.00,0.00,17.86


## Query 11 — Tasa de cancelaciones y devoluciones por categoría

### Pregunta de negocio

¿Qué categorías presentan una mayor proporción de pedidos cancelados o devueltos?

In [22]:
QUERY_11 = f"""
WITH order_categories AS (
    SELECT DISTINCT
        o.order_id,
        o.status,
        c.category_id,
        c.category_name

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.products` AS p
        ON oi.product_id = p.product_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.categories` AS c
        ON p.category_id = c.category_id
)


SELECT
    category_id,
    category_name,
    COUNT(*) AS total_orders,
    COUNTIF(status = 'delivered') AS delivered_orders,
    COUNTIF(status = 'cancelled') AS cancelled_orders,
    COUNTIF(status = 'returned') AS returned_orders,
    
    ROUND(
        SAFE_DIVIDE(COUNTIF(status = 'cancelled'), COUNT(*)) * 100,
        2
    ) AS cancellation_rate_pct,
    
    ROUND(
        SAFE_DIVIDE(COUNTIF(status = 'returned'), COUNTIF(status IN ('delivered', 'returned'))) * 100,
        2
    ) AS return_rate_pct,
    
    ROUND(SAFE_DIVIDE(COUNTIF(status IN ('cancelled', 'returned')), COUNT(*)) * 100, 2) AS incident_rate_pct

FROM
    order_categories

GROUP BY
    category_id,
    category_name

ORDER BY
    incident_rate_pct DESC,
    total_orders DESC
"""

In [23]:
df_query_11 = client.query(QUERY_11).to_dataframe()
df_query_11

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_id,category_name,total_orders,delivered_orders,cancelled_orders,returned_orders,cancellation_rate_pct,return_rate_pct,incident_rate_pct
0,6,Wearables,437,281,25,42,5.72,13.00,15.33
1,4,Audio,250,164,17,21,6.80,11.35,15.20
2,5,Peripherals,310,198,22,25,7.10,11.21,15.16
3,3,Tablets,519,327,30,42,5.78,11.38,13.87
4,8,Storage & Components,691,457,42,51,6.08,10.04,13.46
5,7,Smart Home,626,410,32,37,5.11,8.28,11.02
6,2,Laptops,847,575,40,48,4.72,7.70,10.39
7,1,Smartphones,272,184,13,13,4.78,6.60,9.56


## Query 12 — Comparación entre precio histórico y precio actual

### Pregunta de negocio

¿Cómo ha evolucionado el precio de los productos respecto al precio al que se vendieron históricamente?

In [24]:
QUERY_12 = f"""
WITH product_history AS (
    SELECT
        p.product_id,
        p.sku,
        p.product_name,
        b.brand_name,
        c.category_name,
        p.current_sale_price,
        SUM(oi.quantity) AS units_sold,

        SAFE_DIVIDE(
            SUM(oi.quantity * oi.unit_price), SUM(oi.quantity)
        ) AS avg_historical_unit_price,

        SAFE_DIVIDE(
            SUM(
                oi.quantity
                * oi.unit_price
                * (1 - oi.discount_percent / 100
                )
            ),
            SUM(oi.quantity)
        ) AS avg_effective_unit_price

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.products` AS p
        ON oi.product_id = p.product_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.brands` AS b
        ON p.brand_id = b.brand_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.categories` AS c
        ON p.category_id = c.category_id

    WHERE
        o.status = 'delivered'

    GROUP BY
        p.product_id,
        p.sku,
        p.product_name,
        b.brand_name,
        c.category_name,
        p.current_sale_price
)


SELECT
    product_id,
    sku,
    product_name,
    brand_name,
    category_name,
    units_sold,
    ROUND(current_sale_price, 2) AS current_sale_price,
    ROUND(avg_historical_unit_price, 2) AS avg_historical_unit_price,
    ROUND(avg_effective_unit_price, 2) AS avg_effective_unit_price,
    ROUND(current_sale_price - avg_historical_unit_price, 2) AS price_difference,
    ROUND(
        SAFE_DIVIDE(current_sale_price - avg_historical_unit_price, avg_historical_unit_price) * 100,
        2
    ) AS price_change_pct,

    CASE
        WHEN current_sale_price > avg_historical_unit_price THEN 'Price increased'
        WHEN current_sale_price < avg_historical_unit_price THEN 'Price decreased'
        ELSE 'No change'
    END AS price_trend

FROM
    product_history

ORDER BY
    price_change_pct DESC
"""

In [25]:
df_query_12 = client.query(QUERY_12).to_dataframe()
df_query_12

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_id,sku,product_name,brand_name,category_name,units_sold,current_sale_price,avg_historical_unit_price,avg_effective_unit_price,price_difference,price_change_pct,price_trend
0,5,SKL-08-0005,Reed-Bass X191,Reed-Bass,Storage & Components,47,571.550000000,529.710000000,519.580000000,41.840000000,7.900000000,Price increased
1,2,SKL-07-0002,Alvarez Inc X890,Alvarez Inc,Smart Home,41,345.250000000,320.190000000,311.440000000,25.060000000,7.830000000,Price increased
2,4,SKL-02-0004,SkeletIA A204,SkeletIA,Laptops,55,2384.920000000,2215.150000000,2158.720000000,169.770000000,7.660000000,Price increased
3,45,SKL-01-0045,Adams Ltd X227,Adams Ltd,Smartphones,35,969.110000000,903.650000000,875.630000000,65.460000000,7.240000000,Price increased
4,24,SKL-07-0024,Alvarez Inc M129,Alvarez Inc,Smart Home,69,108.470000000,101.280000000,98.530000000,7.190000000,7.100000000,Price increased
...,...,...,...,...,...,...,...,...,...,...,...,...
65,11,SKL-08-0011,"Grant, Oneill and Medina A398","Grant, Oneill and Medina",Storage & Components,48,209.530000000,201.310000000,193.840000000,8.220000000,4.090000000,Price increased
66,61,SKL-05-0061,Adams Ltd S649,Adams Ltd,Peripherals,44,185.780000000,179.170000000,176.290000000,6.610000000,3.690000000,Price increased
67,54,SKL-06-0054,Alvarez Inc S666,Alvarez Inc,Wearables,52,210.060000000,203.310000000,197.010000000,6.750000000,3.320000000,Price increased
68,32,SKL-08-0032,Johnson-Carter M600,Johnson-Carter,Storage & Components,55,696.310000000,677.230000000,660.730000000,19.080000000,2.820000000,Price increased


## Query 13 — Productos comprados juntos con mayor frecuencia

### Pregunta de negocio

¿Qué combinaciones de productos compran con mayor frecuencia los clientes de SkeletIA dentro del mismo pedido?

In [26]:
QUERY_13 = f"""
WITH delivered_items AS (
    SELECT
        o.order_id,
        oi.product_id

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    WHERE
        o.status = 'delivered'
),

product_pairs AS (
    SELECT
        di1.order_id,
        di1.product_id AS product_a_id,
        di2.product_id AS product_b_id

    FROM
        delivered_items AS di1

    INNER JOIN
        delivered_items AS di2
        ON di1.order_id = di2.order_id
        AND di1.product_id < di2.product_id
),

pair_frequency AS (
    SELECT
        product_a_id,
        product_b_id,
        COUNT(DISTINCT order_id) AS pair_orders

    FROM
        product_pairs

    GROUP BY
        product_a_id,
        product_b_id
),


total_orders AS (
    SELECT
        COUNT(DISTINCT order_id) AS total_delivered_orders

    FROM
        delivered_items
)


SELECT
    pf.product_a_id,
    pa.product_name AS product_a,
    ca.category_name AS category_a,
    pf.product_b_id,
    pb.product_name AS product_b,
    cb.category_name AS category_b,
    pf.pair_orders,

    ROUND(
        SAFE_DIVIDE(pf.pair_orders, tor.total_delivered_orders) * 100,
        2
    ) AS support_pct

FROM
    pair_frequency AS pf

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.products` AS pa
    ON pf.product_a_id = pa.product_id

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.categories` AS ca
    ON pa.category_id = ca.category_id

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.products` AS pb
    ON pf.product_b_id = pb.product_id

INNER JOIN
    `{PROJECT_ID}.{DATASET_ID}.categories` AS cb
    ON pb.category_id = cb.category_id

CROSS JOIN
    total_orders AS tor

ORDER BY
    pf.pair_orders DESC,
    support_pct DESC

LIMIT 20
"""

In [27]:
df_query_13 = client.query(QUERY_13).to_dataframe()
df_query_13

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_a_id,product_a,category_a,product_b_id,product_b,category_b,pair_orders,support_pct
0,28,"Edwards, Murphy and Escobar S420",Tablets,54,Alvarez Inc S666,Wearables,7,0.53
1,17,Alvarez Inc M870,Wearables,68,Chan-Davidson S283,Smartphones,7,0.53
2,43,Reed-Bass S606,Laptops,65,SkeletIA V165,Laptops,6,0.46
3,7,Johnson-Carter A604,Tablets,40,Alvarez Inc M601,Audio,6,0.46
4,4,SkeletIA A204,Laptops,60,Johnson-Carter V650,Smart Home,6,0.46
5,20,Adams Ltd X218,Storage & Components,70,Reed-Bass S923,Laptops,6,0.46
6,49,SkeletIA X980,Storage & Components,52,"Steele, Bowman and Martin S743",Smartphones,6,0.46
7,30,Pope Group S668,Tablets,57,Reed-Bass A424,Laptops,5,0.38
8,43,Reed-Bass S606,Laptops,49,SkeletIA X980,Storage & Components,5,0.38
9,9,"Steele, Bowman and Martin S411",Storage & Components,48,Adams Ltd M179,Smart Home,5,0.38


## Query 14 — Riesgo de rotura de stock según demanda reciente

### Pregunta de negocio

¿Qué productos presentan riesgo de quedarse sin stock si mantienen su ritmo reciente de ventas?

In [28]:
QUERY_14 = f"""
WITH reference_date AS (
    SELECT
        MAX(DATE(order_date)) AS max_order_date

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders`
),

recent_sales AS (
    SELECT
        oi.product_id,
        COUNT(DISTINCT o.order_id) AS orders_90d,
        SUM(oi.quantity) AS units_sold_90d

    FROM
        `{PROJECT_ID}.{DATASET_ID}.orders` AS o

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
        ON o.order_id = oi.order_id

    CROSS JOIN
        reference_date AS rd

    WHERE
        o.status = 'delivered'
        AND DATE(o.order_date) BETWEEN DATE_SUB(rd.max_order_date, INTERVAL 89 DAY)
            AND rd.max_order_date

    GROUP BY
        oi.product_id
),

product_demand AS (
    SELECT
        p.product_id,
        p.sku,
        p.product_name,
        b.brand_name,
        c.category_name,
        p.stock,
        COALESCE(rs.orders_90d, 0) AS orders_90d,
        COALESCE(rs.units_sold_90d, 0) AS units_sold_90d,
        SAFE_DIVIDE(
            COALESCE(rs.units_sold_90d, 0), 90
        ) AS avg_units_per_day

    FROM
        `{PROJECT_ID}.{DATASET_ID}.products` AS p

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.brands` AS b
        ON p.brand_id = b.brand_id

    INNER JOIN
        `{PROJECT_ID}.{DATASET_ID}.categories` AS c
        ON p.category_id = c.category_id

    LEFT JOIN
        recent_sales AS rs
        ON p.product_id = rs.product_id
)


SELECT
    product_id,
    sku,
    product_name,
    brand_name,
    category_name,
    stock,
    orders_90d,
    units_sold_90d,
    ROUND(avg_units_per_day, 2) AS avg_units_per_day,
    ROUND(SAFE_DIVIDE(stock, avg_units_per_day), 1) AS estimated_days_of_stock,

    CASE
        WHEN stock = 0 THEN 'Critical - out of stock'
        WHEN units_sold_90d = 0 THEN 'No recent demand'
        WHEN SAFE_DIVIDE(stock, avg_units_per_day) < 15 THEN 'High risk'
        WHEN SAFE_DIVIDE(stock, avg_units_per_day) < 30 THEN 'Medium risk'
        WHEN SAFE_DIVIDE(stock, avg_units_per_day) < 60 THEN 'Low risk'
        ELSE 'Sufficient stock'
    END AS stock_risk

FROM
    product_demand

ORDER BY
    CASE
        WHEN stock = 0 THEN 1
        WHEN units_sold_90d > 0
            AND SAFE_DIVIDE(stock, avg_units_per_day) < 15 THEN 2
        WHEN units_sold_90d > 0
            AND SAFE_DIVIDE(stock, avg_units_per_day) < 30 THEN 3
        WHEN units_sold_90d > 0
            AND SAFE_DIVIDE(stock, avg_units_per_day) < 60 THEN 4
        WHEN units_sold_90d > 0 THEN 5
        ELSE 6
    END,
    estimated_days_of_stock ASC
"""

In [29]:
df_query_14 = client.query(QUERY_14).to_dataframe()
df_query_14

/Users/dyingskeleton/IA Engineering/repositorios/tc-sql-bometon/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_id,sku,product_name,brand_name,category_name,stock,orders_90d,units_sold_90d,avg_units_per_day,estimated_days_of_stock,stock_risk
0,9,SKL-08-0009,"Steele, Bowman and Martin S411","Steele, Bowman and Martin",Storage & Components,1,14,21,0.23,4.3,High risk
1,63,SKL-02-0063,Pope Group S222,Pope Group,Laptops,3,9,13,0.14,20.8,Medium risk
2,12,SKL-04-0012,"Grant, Oneill and Medina M473","Grant, Oneill and Medina",Audio,29,22,28,0.31,93.2,Sufficient stock
3,58,SKL-07-0058,"Grant, Oneill and Medina M888","Grant, Oneill and Medina",Smart Home,18,16,17,0.19,95.3,Sufficient stock
4,4,SKL-02-0004,SkeletIA A204,SkeletIA,Laptops,33,23,30,0.33,99.0,Sufficient stock
...,...,...,...,...,...,...,...,...,...,...,...
65,22,SKL-08-0022,"Grant, Oneill and Medina X123","Grant, Oneill and Medina",Storage & Components,217,11,14,0.16,1395.0,Sufficient stock
66,42,SKL-08-0042,Pope Group X848,Pope Group,Storage & Components,248,15,16,0.18,1395.0,Sufficient stock
67,35,SKL-04-0035,Johnson-Carter S404,Johnson-Carter,Audio,223,14,14,0.16,1433.6,Sufficient stock
68,34,SKL-02-0034,"Steele, Bowman and Martin A655","Steele, Bowman and Martin",Laptops,210,10,11,0.12,1718.2,Sufficient stock
